In [4]:
import numpy as np
import sys
sys.path.append("/home/steven/thesis/mod")
from model_independent import model_independent
from model_independent import Growth_Factor

In [22]:
from scipy.integrate import trapezoid
c = 299792.458
h0 = 0.735
omh2 = 0.1424
om = omh2/h0**2

ze = 0.5
ae = 1/(1+ze)
a = np.linspace(ae,1.0,200)
model = model_independent(np.array([1]),a,h0,om,-1)

Da = ae*c/h0/100 * trapezoid(1/a**2/model.lcdm_expf(),a)
kh = 0.1
k = kh*h0
ell = Da * k
theta = 2*np.pi/ell*180/np.pi*60
print(theta, ell)

241.9428789305302 89.27727112895137


In [ ]:
## Pure cheb as expansion function
def model (a,*c) :
    h0 = 0.735 # 0.6766
    cheb_poly = np.polynomial.chebyshev.Chebyshev(c, domain=[0.0,1.0])
    da = (1-a[0])/a[0]
    x = (a-a[0])/(a[-1]-a[0])
    expf = da*h0/(1+da*x)**2/(cheb_poly(x))
    return expf/expf[-1]

In [ ]:
## normalized cheb, can be used as shifted cheb by : cheb(c,2x+1)
def cheb(c,a):
    x = np.arccos(a)
    result = 0
    for i in range(len(c)) :
        if i ==0 :
            result += (1/np.pi)**(1/2)*c[i]
        else :
            result += (2/np.pi)**(1/2)*c[i]*np.cos(i*x)
    return result

In [ ]:
## Alex' codes

import numpy as np 
import matplotlib.pyplot as plt 

x = np.linspace(0,1,100)
c = np.array([0.944,-0.357,0.047,0.0052])

a_min = 0.294118
h = 0.735
Om = 0.36

delta_a = (1-a_min)/a_min

def a(x):   
    return (1+delta_a*x)*a_min

def E(x,e): 
    E = delta_a/(1+delta_a*x)**2/e/h
    return  E / E[-1]

#this is what is implemented in the lastro, see also def. in https://scipost.org/SciPostAstro.2.1.001/pdf appendix A
def chebyshev(x, c):
    n = np.arccos(x)
    result = 0
    for i in range(len(c)): 
        if i == 0: 
            result += (1/np.pi)**(1/2)*c[i]
        else: 
            result += (2/np.pi)**(1/2)*c[i]*np.cos(i*n)
    return result

def E_lcdm(a):
    return np.sqrt(Om*a**(-3)+1-Om)

#numpy
cheb = np.polynomial.chebyshev.Chebyshev(c)

plt.figure()
plt.loglog(x, chebyshev(2*x-1,c), label='own impl.')
plt.loglog(x, cheb(2*x-1), label='numpy')
plt.legend()
plt.grid()

plt.figure()
plt.loglog(a(x), E(x,chebyshev(2*x-1,c)), label='own impl.')
# plt.loglog(a(x), E(x,cheb(2*x-1)),label='numpy')
plt.loglog(a(x), E_lcdm(a(x)),label='lcdm')
plt.legend()
plt.grid()
plt.show()


In [ ]:
# n=0 model

def model_expf (self, a=None, c=None) :
        if c is None :
            c = self.c
        if a is not None : # Needed for odeint
            x = (a-self.amin)/(self.amax-self.amin)
        else :
            x = self.x
        expf = self.da/(1+self.da*x)**2/self.chebs(c,2*x-1)/self.h0
        expf = expf/self.norm
        return expf
    
    def d_expansion_function_d_a (self,a=None,c=None):
        if c is None :
            c = self.c
        if a is not None : # Needed for odeint
            x = (a-self.amin)/(self.amax-self.amin)
        else :
            x = self.x
        A = (1+x*self.da)
        prefactor = -self.da/self.h0/(self.a[-1]-self.a[0])/A**2/self.chebs(c,2*x-1)
        term1 = 2*self.da/A
        term2 = 2*self.cheb_derivative(c,2*x-1)/self.chebs(c,2*x-1)
        d_expf = prefactor*(term1 + term2)/self.norm
        return d_expf

In [ ]:
# Testing if the new model is implemented correctly

c = np.array([0.119,-0.0349,0.0032])
c = np.array([-0.110, 0.02561])
a_new = np.linspace(0.001,1,1000)
new_model = model_independent(c,a_new,om=0.32,h0=0.735, n=-1)
new_model2 = model_independent(c,a_new,om=0.37,h0=0.735, n=-1)

plt.loglog(a_new, new_model.model_expf(),label="model")
plt.loglog(a_new, new_model.lcdm_expf(),label="LCDM om=0.32")
plt.loglog(a_new, new_model2.lcdm_expf(),label="LCDM om=0.37")

plt.legend()
plt.show()

test1 = model_independent(c,a,n=-1)
test2 = model_independent(c,a)
plt.plot(a,test1.d_expansion_function_d_a(),label="n=0")
plt.plot(a,test2.d_expansion_function_d_a(),label="n=-1")
plt.legend()

plt.plot(test1.a,test1.model_expf(),label="n=-1")
plt.plot(test1.a,test1.model_expf(n=0),label="n=0")
plt.plot(test1.a,test1.lcdm_expf(),label="lcdm")
plt.legend()
plt.show()

## Testing the implementation of Growth Factor
c = np.array([0.72196921, -0.15288923 ,-0.02065917 , 0.00706415])
zm = np.loadtxt("z_matterme.txt")
am = 1/(1+zm)[::-1]
dplus = Growth_Factor(c,am, n=-1)
dpl_kids = np.loadtxt('dpl_kids.txt')[::-1]
dpl_kids = dpl_kids/dpl_kids[0]
Dplus = []
Dpluss = []

for i in dplus.a :
    Dplus.append(dplus.growth_factor(i))
    # Dpluss.append(dplus.D_plus(i))
# Dpluss = np.array(Dpluss)
# Dpluss = Dpluss/Dpluss[0]
Dplus = np.array(Dplus)
Dplus = Dplus/Dplus[0]
plt.plot(dplus.a,Dplus,label='model',marker='.')
plt.plot(dplus.a,dpl_kids,label='lcdm',marker='.')
# plt.plot(dplus.a,Dpluss,label='model')
plt.legend()
# plt.plot(a,)
plt.show()

c = np.array([1.22204523e+00, -6.32980436e-01,  1.58064939e-01, -4.90222909e-02,
  2.31382692e-02, -1.04376390e-02,  4.24459960e-03, -1.52263287e-03,
  1.13066612e-03])
zm = np.loadtxt("z_matterme.txt")
am = 1/(1+zm)[::-1]
dplus = Growth_Factor(c,am, n=0)
dpl_kids = np.loadtxt('dpl_kids.txt')[::-1]
dpl_kids = dpl_kids/dpl_kids[0]
Dplus = []
Dpluss = []

for i in dplus.a :
    Dplus.append(dplus.growth_factor(i))
    # Dpluss.append(dplus.D_plus(i))
# Dpluss = np.array(Dpluss)
# Dpluss = Dpluss/Dpluss[0]
Dplus = np.array(Dplus)
Dplus = Dplus/Dplus[0]
plt.plot(dplus.a,Dplus,label='model',marker='.')
plt.plot(dplus.a,dpl_kids,label='lcdm',marker='.')
# plt.plot(dplus.a,Dpluss,label='model')
plt.legend()
# plt.plot(a,)
plt.show()

In [ ]:
# Look at the c_ell from all bin pairs

import numpy as np
ells = np.loadtxt("ellsme.txt")
cell = {}
for i in range(5):
    for j in range(i+1) :
        cell[f"{i+1}{j+1}"] = np.loadtxt(f"c_ell_lcdm_{i+1}_{j+1}.txt")
        plt.loglog(ells,cell[f"{i+1}{j+1}"],label=f"bin{i+1}{j+1}")
plt.legend()